# Week 07 과제 · 모범답안 (교수자용)

> 교수자용 모범답안입니다. 학생에게 배포하지 마세요.

## 준비: 자동 확인 도우미

In [1]:
import math
from datetime import date, timedelta

def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")


def raises(exc_type, fn, *args):
    try:
        fn(*args)
    except exc_type:
        return True
    except NotImplementedError:
        raise
    return False

---
## 문제 1. 은행 계좌 클래스 (2점)

`BankAccount(owner, balance=0)`

| 메서드/속성 | 동작 |
|---|---|
| `owner`, `balance` | 예금주, 잔액 |
| `history` | `("입금", 금액)` / `("출금", 금액)` 튜플 리스트. **계좌마다 따로** |
| `deposit(amount)` | 0 이하 금액이면 `ValueError`. 잔액 증가, 기록 추가 |
| `withdraw(amount)` | 0 이하이거나 잔액보다 크면 `ValueError`(잔액·기록 변화 없음). 잔액 감소, 기록 추가 |
| `__str__` | `"김목포님의 잔액: 10,000원"` |

In [2]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("입금액은 0보다 커야 합니다")
        self.balance += amount
        self.history.append(("입금", amount))

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("출금액은 0보다 커야 합니다")
        if amount > self.balance:
            raise ValueError("잔액이 부족합니다")
        self.balance -= amount
        self.history.append(("출금", amount))

    def __str__(self):
        return f"{self.owner}님의 잔액: {self.balance:,}원"

In [3]:
def _scenario():
    acc = BankAccount("김목포")
    acc.deposit(10000)
    acc.withdraw(3000)
    return acc

check("입금·출금 후 잔액", lambda: _scenario().balance == 7000)
check("거래 기록", lambda: _scenario().history == [("입금", 10000), ("출금", 3000)])
check("__str__ 형식", lambda: str(_scenario()) == "김목포님의 잔액: 7,000원")
check("잔액 부족 → ValueError", lambda: raises(ValueError, _scenario().withdraw, 7001))
check("실패한 출금은 잔액·기록 변화 없음", lambda: (lambda a: (raises(ValueError, a.withdraw, 99999), a.balance, len(a.history)) == (True, 7000, 2))(_scenario()))
check("0원 입금 → ValueError", lambda: raises(ValueError, BankAccount("가").deposit, 0))
check("계좌마다 기록이 따로", lambda: (lambda a, b: (a.deposit(1), b.history)[1] == [])(BankAccount("가"), BankAccount("나")))
check("초기 잔액", lambda: BankAccount("가", 5000).balance == 5000)

✅ 통과 - 입금·출금 후 잔액
✅ 통과 - 거래 기록
✅ 통과 - __str__ 형식
✅ 통과 - 잔액 부족 → ValueError
✅ 통과 - 실패한 출금은 잔액·기록 변화 없음
✅ 통과 - 0원 입금 → ValueError
✅ 통과 - 계좌마다 기록이 따로
✅ 통과 - 초기 잔액


---
## 문제 2. 도형 상속과 다형성 (2점)

세 클래스를 만드세요.

| 클래스 | 생성자 | `area()` | `perimeter()` | `name` 속성 |
|---|---|---|---|---|
| `Rectangle` | `(width, height)` | 가로×세로 | 2×(가로+세로) | `"직사각형"` |
| `Square` | `(side)` – **`Rectangle`을 상속** | 상속 그대로 | 상속 그대로 | `"정사각형"` |
| `Circle` | `(radius)` | π r² (소수 둘째 자리 반올림) | 2π r (소수 둘째 자리 반올림) | `"원"` |

그리고 도형 리스트를 받아 넓이 합을 소수 둘째 자리까지 돌려주는 `total_area(shapes)`를 만드세요. (어떤 도형인지 검사하지 말고 `shape.area()`만 호출 → **다형성**)

In [4]:
class Rectangle:
    name = "직사각형"

    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


class Square(Rectangle):
    name = "정사각형"

    def __init__(self, side):
        super().__init__(side, side)


class Circle:
    name = "원"

    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return round(math.pi * self.radius ** 2, 2)

    def perimeter(self):
        return round(2 * math.pi * self.radius, 2)


def total_area(shapes):
    return round(sum(s.area() for s in shapes), 2)

In [5]:
check("Rectangle(3, 4)", lambda: (Rectangle(3, 4).area(), Rectangle(3, 4).perimeter()) == (12, 14))
check("Square는 Rectangle을 상속", lambda: issubclass(Square, Rectangle))
check("Square(5)", lambda: (Square(5).area(), Square(5).perimeter(), Square(5).name) == (25, 20, "정사각형"))
check("Circle(1)", lambda: (Circle(1).area(), Circle(1).perimeter()) == (3.14, 6.28))
check("Circle(2.5) 넓이", lambda: Circle(2.5).area() == 19.63)
check("total_area 다형성", lambda: total_area([Rectangle(3, 4), Square(2), Circle(1)]) == 19.14)
check("빈 리스트 → 0", lambda: total_area([]) == 0)

✅ 통과 - Rectangle(3, 4)
✅ 통과 - Square는 Rectangle을 상속
✅ 통과 - Square(5)
✅ 통과 - Circle(1)
✅ 통과 - Circle(2.5) 넓이
✅ 통과 - total_area 다형성
✅ 통과 - 빈 리스트 → 0


---
## 문제 3. 할 일 목록 클래스 (3점)

`TodoList`

| 메서드 | 동작 |
|---|---|
| `add(title, due=None)` | 할 일을 추가하고 **1부터 시작하는 번호**를 돌려줍니다. 제목이 공백뿐이면 `ValueError` |
| `complete(todo_id)` | 해당 번호를 완료 처리. 없는 번호면 `KeyError`, 이미 완료면 `ValueError` |
| `pending()` | 미완료 할 일의 **제목** 리스트 (추가한 순서) |
| `overdue(today)` | 미완료이면서 `due`가 있고 `due < today`인 제목 리스트 (마감일이 빠른 순) |
| `progress` | `@property`. 완료 비율을 `0.0`~`1.0`으로 (할 일이 없으면 `0.0`) |

> 힌트: 번호 → 할 일 정보(딕셔너리 또는 작은 클래스)를 담는 딕셔너리를 인스턴스 변수로 두세요.

In [6]:
class TodoList:
    def __init__(self):
        self._items = {}
        self._next_id = 1

    def add(self, title, due=None):
        if not title.strip():
            raise ValueError("제목이 비어 있습니다")
        todo_id = self._next_id
        self._items[todo_id] = {"title": title.strip(), "due": due, "done": False}
        self._next_id += 1
        return todo_id

    def complete(self, todo_id):
        item = self._items[todo_id]
        if item["done"]:
            raise ValueError("이미 완료된 할 일입니다")
        item["done"] = True

    def pending(self):
        return [it["title"] for it in self._items.values() if not it["done"]]

    def overdue(self, today):
        late = [it for it in self._items.values()
                if not it["done"] and it["due"] is not None and it["due"] < today]
        return [it["title"] for it in sorted(late, key=lambda it: it["due"])]

    @property
    def progress(self):
        if not self._items:
            return 0.0
        done = sum(it["done"] for it in self._items.values())
        return done / len(self._items)

In [7]:
def _todo():
    t = TodoList()
    t.add("과제 제출", date(2026, 10, 10))      # 1
    t.add("발표 준비", date(2026, 10, 5))       # 2
    t.add("책 반납")                            # 3
    t.add("팀 회의", date(2026, 10, 20))        # 4
    t.complete(4)
    return t

check("번호는 1부터", lambda: TodoList().add("첫 할 일") == 1)
check("pending 순서", lambda: _todo().pending() == ["과제 제출", "발표 준비", "책 반납"])
check("overdue 마감일 순", lambda: _todo().overdue(date(2026, 10, 12)) == ["발표 준비", "과제 제출"])
check("overdue 경계(마감일 당일은 아님)", lambda: _todo().overdue(date(2026, 10, 5)) == [])
check("progress", lambda: _todo().progress == 0.25)
check("빈 목록 progress 0.0", lambda: TodoList().progress == 0.0)
check("없는 번호 → KeyError", lambda: raises(KeyError, _todo().complete, 99))
check("중복 완료 → ValueError", lambda: raises(ValueError, _todo().complete, 4))
check("빈 제목 → ValueError", lambda: raises(ValueError, TodoList().add, "   "))
check("목록끼리 독립", lambda: (lambda a, b: (a.add("x"), b.pending())[1] == [])(TodoList(), TodoList()))

✅ 통과 - 번호는 1부터
✅ 통과 - pending 순서
✅ 통과 - overdue 마감일 순
✅ 통과 - overdue 경계(마감일 당일은 아님)
✅ 통과 - progress
✅ 통과 - 빈 목록 progress 0.0
✅ 통과 - 없는 번호 → KeyError
✅ 통과 - 중복 완료 → ValueError
✅ 통과 - 빈 제목 → ValueError
✅ 통과 - 목록끼리 독립


---
## 문제 4. 스터디룸 예약 시스템 (AI 협업, 3점)

도서관 스터디룸 예약을 관리하는 두 클래스를 만드세요. **AI에게 설계를 먼저 물어보고**, 받은 코드를 자동 확인으로 검증한 뒤 고치세요.

**`Student(name)`**: `name`, `reservations`(자기 예약 번호 리스트, **학생마다 따로**)

**`StudyRoomSystem(rooms)`**: `rooms`는 방 이름 리스트

| 메서드 | 규칙 |
|---|---|
| `register(student)` | 학생 등록 |
| `reserve(name, room, day, hour)` | 예약 번호(1부터)를 돌려줍니다. ① 등록 안 된 학생·없는 방 → `KeyError` ② 운영 시간 9~20시(시작 시각 기준, `hour`가 9 이상 20 이하)를 벗어나면 `ValueError` ③ 같은 방·날짜·시간이 이미 예약됨 → `ValueError` ④ 한 학생은 **같은 날 최대 2건** → `ValueError` |
| `cancel(reservation_id)` | 예약 취소. 없는 번호 → `KeyError`. 취소하면 그 시간은 다시 예약 가능, 학생의 하루 한도도 돌아옵니다. |
| `schedule(room, day)` | 그 방·날짜의 예약 `(hour, name)` 리스트, 시간순 |

In [8]:
class Student:
    def __init__(self, name):
        self.name = name
        self.reservations = []


class StudyRoomSystem:
    MAX_PER_DAY = 2
    OPEN, CLOSE = 9, 20

    def __init__(self, rooms):
        self.rooms = set(rooms)
        self.students = {}
        self.bookings = {}          # id -> (name, room, day, hour)
        self._next_id = 1

    def register(self, student):
        self.students[student.name] = student

    def reserve(self, name, room, day, hour):
        student = self.students[name]
        if room not in self.rooms:
            raise KeyError(room)
        if not self.OPEN <= hour <= self.CLOSE:
            raise ValueError(f"운영 시간 아님: {hour}시")
        for (_n, r, d, h) in self.bookings.values():
            if (r, d, h) == (room, day, hour):
                raise ValueError("이미 예약된 시간입니다")
        same_day = sum(1 for (n, _r, d, _h) in self.bookings.values() if n == name and d == day)
        if same_day >= self.MAX_PER_DAY:
            raise ValueError("하루 예약 한도 초과")
        rid = self._next_id
        self._next_id += 1
        self.bookings[rid] = (name, room, day, hour)
        student.reservations.append(rid)
        return rid

    def cancel(self, reservation_id):
        name, _room, _day, _hour = self.bookings.pop(reservation_id)
        self.students[name].reservations.remove(reservation_id)

    def schedule(self, room, day):
        return sorted((h, n) for (n, r, d, h) in self.bookings.values() if r == room and d == day)

In [9]:
D1, D2 = date(2026, 10, 1), date(2026, 10, 2)

def _system():
    s = StudyRoomSystem(["A", "B"])
    for n in ["가", "나"]:
        s.register(Student(n))
    return s

def _scenario4():
    s = _system()
    r1 = s.reserve("가", "A", D1, 10)
    r2 = s.reserve("나", "A", D1, 9)
    r3 = s.reserve("가", "B", D1, 15)
    return s, (r1, r2, r3)

check("예약 번호 1, 2, 3", lambda: _scenario4()[1] == (1, 2, 3))
check("schedule 시간순", lambda: _scenario4()[0].schedule("A", D1) == [(9, "나"), (10, "가")])
check("같은 방·시간 중복 → ValueError", lambda: raises(ValueError, _scenario4()[0].reserve, "나", "A", D1, 10))
check("하루 2건 초과 → ValueError", lambda: raises(ValueError, _scenario4()[0].reserve, "가", "B", D1, 16))
check("다른 날은 가능", lambda: _scenario4()[0].reserve("가", "B", D2, 16) == 4)
check("운영 시간 경계 9·20 허용, 8·21 거부", lambda: (_system().reserve("가", "A", D1, 20) == 1
      and raises(ValueError, _system().reserve, "가", "A", D1, 8) and raises(ValueError, _system().reserve, "가", "A", D1, 21)))
check("미등록 학생·없는 방 → KeyError", lambda: raises(KeyError, _system().reserve, "다", "A", D1, 10)
      and raises(KeyError, _system().reserve, "가", "Z", D1, 10))


def _cancel_restores():
    s, (r1, r2, r3) = _scenario4()
    s.cancel(r1)
    ok1 = s.reserve("나", "A", D1, 10) == 4          # 그 시간이 다시 비었다
    ok2 = s.reserve("가", "B", D1, 16) == 5          # 하루 한도도 돌아왔다
    return ok1 and ok2 and s.students["가"].reservations == [r3, 5]

check("취소하면 시간·한도 복구", _cancel_restores)
check("없는 예약 취소 → KeyError", lambda: raises(KeyError, _system().cancel, 42))
check("학생마다 예약 목록이 따로", lambda: (lambda s: s.students["나"].reservations == [2])(_scenario4()[0]))

✅ 통과 - 예약 번호 1, 2, 3
✅ 통과 - schedule 시간순
✅ 통과 - 같은 방·시간 중복 → ValueError
✅ 통과 - 하루 2건 초과 → ValueError
✅ 통과 - 다른 날은 가능
✅ 통과 - 운영 시간 경계 9·20 허용, 8·21 거부
✅ 통과 - 미등록 학생·없는 방 → KeyError
✅ 통과 - 취소하면 시간·한도 복구
✅ 통과 - 없는 예약 취소 → KeyError
✅ 통과 - 학생마다 예약 목록이 따로


### AI 활용 기록 (예시)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 | (예) Claude |
| AI가 제안한 설계 | `Student`는 자기 예약 번호만, `StudyRoomSystem`이 방·예약·규칙 검사를 모두 담당 |
| 실패한 항목 | `Student`의 `reservations = []`를 클래스 변수로 써서 학생끼리 목록 공유, 운영 시간을 `9 <= hour < 20`으로 써서 20시 거부, 취소 시 학생 목록에서 번호를 지우지 않음 |
| 직접 고친 부분 | `__init__`에서 `self.reservations = []`, 경계를 `<= 20`으로, `cancel`에서 양쪽 상태를 함께 갱신 |

> **채점 포인트**: 상태가 두 곳(시스템·학생)에 있을 때 **양쪽을 함께 갱신**했는지, 경계(9시·20시)를 확인했는지.

---
## 성찰 (필수)

1. 클래스를 설계할 때 "어떤 클래스가 어떤 상태와 책임을 가질지"를 어떻게 정했나요?
2. 상태가 이어지는 코드를 테스트할 때 어려웠던 점은?
3. AI의 설계 제안 중 받아들인 것과 바꾼 것은?

(학생 작성 영역)